# YOLO26 Solar Filament GPU Training

Stage 2 of the pipeline. Attach the output of yolo26-crop-preprocessing.ipynb as a Kaggle input. This notebook only reads the prepared crops and trains the segmentation model; it does not perform preprocessing.

Set the Kaggle accelerator to GPU T4 x2 before starting. Both visible GPUs are selected, with a global batch of eight images per GPU. Training starts only after the preprocessed handoff has been located and checked.


## 1. Training and handoff configuration


In [ ]:
from pathlib import Path
import os,json,platform,shutil,time

PREPROCESSING_ARTIFACT_DIR=None  # Set explicitly if automatic Kaggle Input discovery finds more than one bundle.
WORKING_ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
TRAINING_ARTIFACT_DIR=WORKING_ROOT/'yolo26_crop_training_artifacts'
RUNS_DIR=TRAINING_ARTIFACT_DIR/'runs'
RUN_NAME='yolo26s_crop_training'
MODEL_NAME='yolo26s-seg.pt'
NUM_EPOCHS=100
BATCH_PER_GPU=8
CPU_LOADER_WORKERS=4
CACHE_MODE=False
DETERMINISTIC=True
SMOKE_TEST=False
SMOKE_TEST_EPOCHS=1
REQUIRE_TWO_GPUS=True
SEED=42
CROP_SIZE=640


## 2. GPU training environment


In [ ]:
for variable in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[variable]='1'
try:
    import cv2,matplotlib.pyplot as plt,numpy as np,pandas as pd,torch,ultralytics
    from IPython.display import display
    from threadpoolctl import threadpool_limits
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError(f'Missing training dependency {exc}; install a YOLO26-capable Ultralytics build.') from exc
torch.manual_seed(SEED)
torch.set_num_threads(1)
try:torch.set_num_interop_threads(1)
except RuntimeError:pass
THREADPOOL_LIMITER=threadpool_limits(limits=1)
GPU_COUNT=torch.cuda.device_count() if torch.cuda.is_available() else 0
if REQUIRE_TWO_GPUS and GPU_COUNT<2:
    raise RuntimeError(f'This training notebook expects two visible GPUs; found {GPU_COUNT}. Select GPU T4 x2 in Kaggle.')
DEVICE=list(range(GPU_COUNT)) if GPU_COUNT>1 else (0 if GPU_COUNT==1 else 'cpu')
BATCH_SIZE=BATCH_PER_GPU*max(1,GPU_COUNT)
NUM_WORKERS=max(1,CPU_LOADER_WORKERS//max(1,GPU_COUNT)) if GPU_COUNT else 0
print('Python',platform.python_version(),'PyTorch',torch.__version__,'CUDA',torch.version.cuda)
print('Ultralytics',ultralytics.__version__)
print(f'GPU count: {GPU_COUNT} | device: {DEVICE} | batch: {BATCH_SIZE} global ({BATCH_PER_GPU}/GPU)')
print(f'Data loader workers: {NUM_WORKERS} per GPU')
for i in range(GPU_COUNT):
    props=torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}, {props.total_memory/2**30:.1f} GiB')


## 3. Load and validate the preprocessing handoff


In [ ]:
def artifact_bundle_is_valid(root):
    root=Path(root)
    return ((root/'prepared_dataset'/'dataset.yaml').is_file()
        and (root/'split_manifest.csv').is_file()
        and (root/'crop_manifest.csv').is_file()
        and (root/'prepared_dataset'/'images'/'train').is_dir()
        and (root/'prepared_dataset'/'images'/'val').is_dir()
        and (root/'prepared_dataset'/'labels'/'train').is_dir()
        and (root/'prepared_dataset'/'labels'/'val').is_dir())

def find_preprocessing_artifact_dir(explicit=None):
    if explicit is not None:
        path=Path(explicit).expanduser().resolve()
        if path.name=='prepared_dataset':path=path.parent
        if not artifact_bundle_is_valid(path):
            raise FileNotFoundError(f'Incomplete preprocessing handoff: {path}')
        return path
    roots=[Path('/kaggle/input'),WORKING_ROOT,Path.cwd()]
    candidates=set()
    for parent in roots:
        if not parent.exists():continue
        if artifact_bundle_is_valid(parent):candidates.add(parent.resolve())
        try:
            for yaml_path in parent.rglob('dataset.yaml'):
                if yaml_path.parent.name=='prepared_dataset':
                    root=yaml_path.parent.parent
                    if artifact_bundle_is_valid(root):candidates.add(root.resolve())
        except OSError:
            pass
    candidates=sorted(candidates)
    if not candidates:
        raise FileNotFoundError('Could not find a complete yolo26_crop_training_artifacts preprocessing bundle under Kaggle Input. Attach the output from yolo26-crop-preprocessing.ipynb or set PREPROCESSING_ARTIFACT_DIR.')
    if len(candidates)>1:
        raise RuntimeError('Multiple preprocessing bundles found; set PREPROCESSING_ARTIFACT_DIR:\n'+'\n'.join(map(str,candidates)))
    return candidates[0]

PREPROCESSING_ARTIFACT_DIR=find_preprocessing_artifact_dir(PREPROCESSING_ARTIFACT_DIR)
YOLO_DATA_DIR=PREPROCESSING_ARTIFACT_DIR/'prepared_dataset'
split_manifest=pd.read_csv(PREPROCESSING_ARTIFACT_DIR/'split_manifest.csv',
    dtype={'file_name':str,'image_id':str,'annotation_view_id':str,'split':str})
crop_manifest=pd.read_csv(PREPROCESSING_ARTIFACT_DIR/'crop_manifest.csv',
    dtype={'file_name':str,'physical_filename':str,'split':str})
required={'file_name','image_id','annotation_view_id','split'}
if not required.issubset(split_manifest.columns):
    raise ValueError(f'Split manifest lacks columns: {sorted(required-set(split_manifest.columns))}')
if split_manifest.image_id.duplicated().any():
    raise ValueError('Split manifest contains duplicate annotation-view IDs.')
if set(split_manifest.split)!={'train','val'}:
    raise ValueError(f'Expected train and val splits, got {sorted(set(split_manifest.split))}')
if split_manifest.groupby('file_name').split.nunique().max()!=1:
    raise ValueError('Physical source images leak across train and validation splits.')
if set(crop_manifest.split)!={'train','val'}:
    raise ValueError('Crop manifest must contain train and validation crops.')
if not (crop_manifest.crop_size==CROP_SIZE).all():
    raise ValueError('Crop manifest crop size does not match training configuration.')
split_identifier=__import__('hashlib').sha256(
    split_manifest[['file_name','split']].drop_duplicates().sort_values('file_name').to_csv(index=False).encode('utf-8')
).hexdigest()[:16]
preprocessing_config_path=PREPROCESSING_ARTIFACT_DIR/'preprocessing_config.json'
if preprocessing_config_path.is_file():
    preprocessing_config=json.loads(preprocessing_config_path.read_text(encoding='utf-8'))
    saved_identifier=preprocessing_config.get('split_identifier')
    if saved_identifier and saved_identifier!=split_identifier:
        raise ValueError(f'Split identifier mismatch: {saved_identifier} != {split_identifier}')
TRAINING_ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
for filename in ['split_manifest.csv','crop_manifest.csv','annotation_mapping.json',
                 'annotation_view_mapping.csv','crop_config.json','preprocessing_config.json',
                 'dataset_audit.json','polygon_iou_sample.json','environment.json']:
    source=PREPROCESSING_ARTIFACT_DIR/filename
    if not source.is_file():source=YOLO_DATA_DIR/filename
    if source.is_file() and source.resolve()!= (TRAINING_ARTIFACT_DIR/filename).resolve():
        shutil.copy2(source,TRAINING_ARTIFACT_DIR/filename)
DATA_YAML=TRAINING_ARTIFACT_DIR/'training_dataset.yaml'
DATA_YAML.write_text(
    f"path: '{YOLO_DATA_DIR.as_posix()}'\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
    encoding='utf-8')
print('Preprocessing bundle:',PREPROCESSING_ARTIFACT_DIR)
print('Prepared crops:',YOLO_DATA_DIR)
print(f'Split {split_identifier} | crops: {len(crop_manifest):,} | train: {(crop_manifest.split=="train").sum():,} | val: {(crop_manifest.split=="val").sum():,}')


## 4. Train YOLO26 on both GPUs

Validation and competition scoring are disabled in this stage. Kaggle GPU time is spent on model training only.


In [ ]:
training_config={
 'model':MODEL_NAME,'task':'segment','class_names':['filament'],'epochs':NUM_EPOCHS,
 'batch':BATCH_SIZE,'batch_per_gpu':BATCH_PER_GPU,'workers_per_gpu':NUM_WORKERS,
 'gpu_count':GPU_COUNT,'device':DEVICE,'imgsz':CROP_SIZE,'cache':CACHE_MODE,
 'pretrained':True,'optimizer':'auto','save':True,'plots':True,'amp':GPU_COUNT>0,
 'run_name':RUN_NAME,'seed':SEED,'deterministic':DETERMINISTIC,
 'validation_enabled':False,'split_identifier':split_identifier,
 'cpu_crop_workers':preprocessing_config.get('cpu_crop_workers') if 'preprocessing_config' in globals() else None,
 'augmentation':{'fliplr':.5,'flipud':.5,'degrees':0,'translate':0,'scale':0,'shear':0,
  'perspective':0,'hsv_h':0,'hsv_s':0,'hsv_v':.15,'mosaic':0,'mixup':0,'copy_paste':0}}
(TRAINING_ARTIFACT_DIR/'training_config.json').write_text(json.dumps(training_config,indent=2),encoding='utf-8')
environment={'python_version':platform.python_version(),'pytorch_version':str(torch.__version__),
 'cuda_version':torch.version.cuda,'ultralytics_version':str(ultralytics.__version__),
 'cuda_available':bool(torch.cuda.is_available()),'gpu_count':GPU_COUNT,'device_selected':DEVICE,
 'gpu_devices':[{'index':i,'name':torch.cuda.get_device_name(i),
                 'total_memory_bytes':int(torch.cuda.get_device_properties(i).total_memory)}
                for i in range(GPU_COUNT)],
 'cpu_loader_workers_per_gpu':NUM_WORKERS,'cpu_crop_workers':training_config['cpu_crop_workers']}
(TRAINING_ARTIFACT_DIR/'training_environment.json').write_text(json.dumps(environment,indent=2),encoding='utf-8')

print('Loading configured model:',MODEL_NAME)
try:
    model=YOLO(MODEL_NAME)
except Exception as exc:
    raise RuntimeError(f'Could not load {MODEL_NAME}; verify YOLO26-capable Ultralytics and checkpoint availability.') from exc
if getattr(model,'task',None)!='segment':
    raise RuntimeError(f'Expected segmentation task, got {getattr(model,"task",None)!r}.')
if crop_manifest.loc[crop_manifest.split=='train'].empty or crop_manifest.loc[crop_manifest.split=='val'].empty:
    raise ValueError('Both crop partitions must have crops.')
active_epochs=SMOKE_TEST_EPOCHS if SMOKE_TEST else NUM_EPOCHS
active_name=f'{RUN_NAME}_smoke' if SMOKE_TEST else RUN_NAME
kwargs={'data':str(DATA_YAML),'epochs':active_epochs,'imgsz':CROP_SIZE,'batch':BATCH_SIZE,
 'device':DEVICE,'workers':NUM_WORKERS,'cache':CACHE_MODE,'project':str(RUNS_DIR),
 'name':active_name,'exist_ok':True,'pretrained':True,'optimizer':'auto','seed':SEED,
 'deterministic':DETERMINISTIC,'val':False,'save':True,'plots':True,'verbose':True,'amp':GPU_COUNT>0,
 'fliplr':.5,'flipud':.5,'degrees':0,'translate':0,'scale':0,'shear':0,'perspective':0,
 'hsv_h':0,'hsv_s':0,'hsv_v':.15,'mosaic':0,'mixup':0,'copy_paste':0}
print(json.dumps({k:v for k,v in kwargs.items() if k!='data'},indent=2))
(TRAINING_ARTIFACT_DIR/'training_config_used.json').write_text(json.dumps(kwargs,indent=2),encoding='utf-8')
train_start=time.perf_counter()
train_results=model.train(**kwargs)
training_seconds=time.perf_counter()-train_start
run_dir=Path(model.trainer.save_dir)
last_checkpoint=run_dir/'weights'/'last.pt'
if not last_checkpoint.is_file():raise FileNotFoundError(f'Mandatory last.pt missing: {last_checkpoint}')
exported_checkpoint=None
if not SMOKE_TEST:
    exported_checkpoint=TRAINING_ARTIFACT_DIR/'yolo26s_crop_training.pt'
    shutil.copy2(last_checkpoint,exported_checkpoint)
    best=run_dir/'weights'/'best.pt'
    if best.is_file():shutil.copy2(best,TRAINING_ARTIFACT_DIR/'best_unvalidated.pt')
print(f'Training duration: {training_seconds/60:.1f} min')
print('Last checkpoint:',last_checkpoint,'export:',exported_checkpoint)
print('best.pt is not validation-selected because val=False.')


## 5. Training diagnostics and export record


In [ ]:
results_csv=run_dir/'results.csv'
results_plot=run_dir/'results.png'
if results_csv.is_file():
    history=pd.read_csv(results_csv)
    history.columns=[c.strip() for c in history.columns]
    display(history.tail())
    losses=[c for c in history.columns if 'loss' in c.lower()]
    if losses:
        ax=history[losses].plot(figsize=(10,5),title='YOLO26 crop training losses')
        ax.set_xlabel('Epoch');ax.set_ylabel('Loss')
        plt.tight_layout();plt.show()
    history.to_csv(TRAINING_ARTIFACT_DIR/'training_results.csv',index=False)
    shutil.copy2(results_csv,TRAINING_ARTIFACT_DIR/'ultralytics_results.csv')
if results_plot.is_file():shutil.copy2(results_plot,TRAINING_ARTIFACT_DIR/'ultralytics_results.png')
run_record={'trained_utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
 'training_seconds':training_seconds,'run_directory':str(run_dir.resolve()),
 'last_checkpoint':str(last_checkpoint.resolve()),
 'exported_checkpoint':str(exported_checkpoint.resolve()) if exported_checkpoint else None,
 'smoke_test':SMOKE_TEST,'epochs_requested':active_epochs,'validation_enabled':False,
 'ultralytics_version':ultralytics.__version__,'python_version':platform.python_version(),
 'pytorch_version':str(torch.__version__),'cuda_version':torch.version.cuda,
 'device':DEVICE,'gpu_count':GPU_COUNT,'split_identifier':split_identifier,
 'dataset_yaml':str(DATA_YAML.resolve())}
(TRAINING_ARTIFACT_DIR/'training_run.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print('Post-processing handoff:',TRAINING_ARTIFACT_DIR)


## Stage handoff

Save yolo26_crop_training_artifacts as a Kaggle output dataset or attach the saved output to yolo-post-processing.ipynb. The training artifact root contains yolo26s_crop_training.pt, split_manifest.csv, crop_manifest.csv, training_config.json, annotation mappings, and run metadata.

Run yolo-post-processing.ipynb later with the Kaggle accelerator set to None. That notebook uses CPU inference and a four-thread CPU limit.
